<a href="https://colab.research.google.com/github/juanpablor69/Proyecto_IA/blob/main/02_preprocesado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto - Inteligencia Artificial para las Ciencias y las Ingenierías**
##Entrega 2

Autor: Juan Pablo Rendón Jimenez. \
Universidad de Antioquia

Las Pruebas Saber Pro son exámenes estandarizados que se administran en Colombia para evaluar la calidad y el nivel de conocimiento y competencias de los estudiantes de educación superior, es decir, de instituciones de educación superior como universidades y tecnológicos. Hace parte de los esfuerzos del Gobierno de Colombia para monitorear y mejorar la calidad de la educación superior en el país.

Estas Pruebas constan cinco componentes genéricos, Inglés, Lectura Crítica, Competencias Ciudadanas, Razonamiento Cuantitativo y Comunicación Escrita.

El objetivo de este trabajo será crear un modelo de clasificación que para cada estudiante prediga qué desempeño va a tener: bajo, medio-bajo, medio-alto o alto.


## Enlace con Kaggle

In [2]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '.'
!chmod 600 ./kaggle.json
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 1.38GB/s]


## Lectura e inspeccion de datos

In [3]:
!unzip udea*.zip > /dev/null
!wc *.csv

   296787    296787   4716673 submission_example.csv
   296787   4565553  59185238 test.csv
   692501  10666231 143732437 train.csv
  1286075  15528571 207634348 total


In [4]:
import pandas as pd
import numpy as np

datos = pd.read_csv("train.csv")
print("Dimensiones del dataset:", datos.shape)

Dimensiones del dataset: (692500, 21)


La base de datos contiene 692500 filas y 21 columnas. La estructura es la siguiente:

In [ ]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


## Analisis de la base de datos:

In [ ]:
print(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ID                           692500 non-null  int64  
 1   PERIODO_ACADEMICO            692500 non-null  int64  
 2   E_PRGM_ACADEMICO             692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD  686213 non-null  object 
 5   E_HORASSEMANATRABAJA         661643 non-null  object 
 6   F_ESTRATOVIVIENDA            660363 non-null  object 
 7   F_TIENEINTERNET              665871 non-null  object 
 8   F_EDUCACIONPADRE             669322 non-null  object 
 9   F_TIENELAVADORA              652727 non-null  object 
 10  F_TIENEAUTOMOVIL             648877 non-null  object 
 11  E_PRIVADO_LIBERTAD           692500 non-null  object 
 12  E_PAGOMATRICULAPROPIO        686002 non-null  object 
 13 

In [ ]:
print("\n--- Valores faltantes por columna ---")
print(datos.isnull().sum())


--- Valores faltantes por columna ---
ID                                 0
PERIODO_ACADEMICO                  0
E_PRGM_ACADEMICO                   0
E_PRGM_DEPARTAMENTO                0
E_VALORMATRICULAUNIVERSIDAD     6287
E_HORASSEMANATRABAJA           30857
F_ESTRATOVIVIENDA              32137
F_TIENEINTERNET                26629
F_EDUCACIONPADRE               23178
F_TIENELAVADORA                39773
F_TIENEAUTOMOVIL               43623
E_PRIVADO_LIBERTAD                 0
E_PAGOMATRICULAPROPIO           6498
F_TIENECOMPUTADOR              38103
F_TIENEINTERNET.1              26629
F_EDUCACIONMADRE               23664
RENDIMIENTO_GLOBAL                 0
INDICADOR_1                        0
INDICADOR_2                        0
INDICADOR_3                        0
INDICADOR_4                        0
dtype: int64


## Limpieza de datos
Luego del analisis realizado identificamos dos columnas completamente identicas (Internet), por lo tanto procederemos con su eliminacion. Ademas, observamos un alto numero de valores faltantes, por lo que rellenaremos esos espacios con un NO REPORTA para evitar errores o sesgos.

In [6]:
# Elimino columna duplicada
if 'F_TIENEINTERNET.1' in datos.columns:
    datos.drop(columns=['F_TIENEINTERNET.1'], inplace=True)

# IMPUTACION DE DATOS FALTANTES
cols_categoricas = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
    'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
    'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PAGOMATRICULAPROPIO',
    'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'
]

for col in cols_categoricas:
    if col in datos.columns:
        datos[col] = datos[col].fillna('no info')

columnas_categoricas = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA',
    'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL', 'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'
]

for col in columnas_categoricas:
    datos[col] = datos[col].fillna('no info')

datos.fillna(0, inplace=True)

# === Mapeo ordinal: Valor matrícula ===
mapa_matricula = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0,
    'no info': -1
}
datos['E_VALORMATRICULAUNIVERSIDAD'] = datos['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)

datos['F_EDUCACIONMADRE'] = datos['F_EDUCACIONMADRE'].replace(
    ['No sabe', 'No Aplica'], 'no info'
)



## Conversion de columnas en one-hot

En la entrega 3 usaremos modelos basados en Random Forest, entre otros. Para hacer uso de estos modelos necesitaremos convertir algunos datos ya que ellos solo trabajan con valores numéricos. En este caso vamos a reemplazar los datos como "Si" o "No" por una variable booleana como 0 y 1.

In [ ]:
# === One-hot encoding automático ===
datos = pd.get_dummies(datos, columns=['F_EDUCACIONMADRE'], prefix='EDUMADRE')

# === Confirmar cambios ===
print("\n--- Limpieza completada ---")
print(datos.info())


--- Limpieza completada ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 30 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   ID                                             692500 non-null  int64  
 1   PERIODO_ACADEMICO                              692500 non-null  int64  
 2   E_PRGM_ACADEMICO                               692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO                            692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD                    692500 non-null  float64
 5   E_HORASSEMANATRABAJA                           692500 non-null  object 
 6   F_ESTRATOVIVIENDA                              692500 non-null  object 
 7   F_TIENEINTERNET                                692500 non-null  object 
 8   F_EDUCACIONPADRE                               692500 non-null  object 
 9   F_TIENEL

Como vemos en la tabla, tenemos toda la base de datos sin incongruencias ni valores faltantes.

In [ ]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,EDUMADRE_Educación profesional incompleta,EDUMADRE_Ninguno,EDUMADRE_Postgrado,EDUMADRE_Primaria completa,EDUMADRE_Primaria incompleta,EDUMADRE_Secundaria (Bachillerato) completa,EDUMADRE_Secundaria (Bachillerato) incompleta,EDUMADRE_Técnica o tecnológica completa,EDUMADRE_Técnica o tecnológica incompleta,EDUMADRE_no info
0,904256,20212,ENFERMERIA,BOGOTÁ,6.25,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,False,False,True,False,False,False,False,False,False,False
1,645256,20212,DERECHO,ATLANTICO,3.25,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,False,False,False,False,False,False,False,False,True,False
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,3.25,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,False,False,False,False,False,True,False,False,False,False
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,4.75,0,Estrato 4,Si,No sabe,Si,...,False,False,False,False,False,True,False,False,False,False
4,989032,20212,PSICOLOGIA,ANTIOQUIA,3.25,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,False,False,False,True,False,False,False,False,False,False
